# Fused LoRA Triton Kernels — Correctness & Benchmark

Self-contained notebook to test custom fused LoRA kernels.

**What this tests:**
1. `FusedLoRALinear` — fused forward+backward for a single LoRA-augmented linear layer
2. `FusedLoRAQKV` — groups Q/K/V LoRA projections into a single autograd function
3. Numerical correctness vs unfused baseline (forward + gradients)
4. Memory comparison (peak VRAM)
5. Speed comparison (wall-clock)

**Requirements (use pinned versions):**  
- **PyTorch 2.10.x has a memory leak** (~15 GB/step on multi-GPU) — do not use. Use **torch 2.7.1+cu128** (triton comes as its dependency; full training stack pins triton 3.6.0 separately).

- **For this notebook only (Colab or quick local test):** use `requirements-notebook.txt` (no flash-linear-attention build):
  ```bash
  pip install -r requirements-notebook.txt --index-url https://download.pytorch.org/whl/cu128
  ```
  Or from the notebook folder: `pip install -r ../requirements-notebook.txt --index-url ...`

- **For full training (p4de):** use `requirements-pinned.txt` in a proper env where flash-linear-attention can be built (CUDA, ninja, etc.). Do **not** use the full file on Colab — FLA build often fails there.

**Run on:** Colab Free (T4) for correctness, Colab Pro (A100) for realistic benchmarks, or p4de with pinned env.

In [ ]:
# Optional: run this first on Colab to install pinned torch (avoids PyTorch 2.10 memory leak).
# Triton is installed as torch's dependency (3.3.1). Skip if you already have the pinned stack.
!pip install torch==2.7.1+cu128 torchvision==0.22.1+cu128 --index-url https://download.pytorch.org/whl/cu128

# After this cell finishes, restart the runtime (Runtime → Restart session), then run the rest of the notebook.

In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import math
import time

# Pinned stack: torch 2.7.1+cu128, triton 3.6.0 (see requirements-pinned.txt)
# PyTorch 2.10.x has a known memory leak — do not use for training.
_torch_ver = getattr(torch, "__version__", "0.0.0")
if _torch_ver.startswith("2.10"):
    raise RuntimeError(
        f"PyTorch {_torch_ver} has a memory leak. Use torch 2.7.1+cu128 from requirements-pinned.txt."
    )
print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
if torch.cuda.is_available():
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    total_mem = getattr(torch.cuda.get_device_properties(0), 'total_memory',
                        getattr(torch.cuda.get_device_properties(0), 'total_mem', 0))
    print(f"VRAM: {total_mem / 1e9:.1f} GB")

try:
    import triton
    import triton.language as tl
    HAS_TRITON = True
    print(f"Triton: {triton.__version__}")
except ImportError:
    HAS_TRITON = False
    print("Triton not available — will use PyTorch fallback")

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
DTYPE = torch.bfloat16

## 1. Baseline: Standard (Unfused) LoRA

This matches the current `LoRALinear` implementation in `lora_utils.py`.

In [ ]:
class StandardLoRALinear(nn.Module):
    """Unfused LoRA: base linear + two separate matmuls for LoRA path."""

    def __init__(self, in_features, out_features, rank, alpha, dtype=torch.bfloat16):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank

        self.linear = nn.Linear(in_features, out_features, bias=False, dtype=dtype)
        self.linear.weight.requires_grad = False  # frozen base

        self.lora_A = nn.Parameter(torch.empty(rank, in_features, dtype=dtype))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank, dtype=dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        base_out = self.linear(x)
        lora_out = F.linear(F.linear(x, self.lora_A), self.lora_B)
        return base_out + lora_out * self.scaling

## 2. Fused LoRA — Custom Autograd Function

Key optimizations vs the standard path:
- **Forward**: computes `base + scale * B(A(x))` but does NOT store the intermediate `A(x)` tensor
- **Backward**: recomputes `A(x)` from saved `x` and `A`, trading compute for memory
- Saves input `x` only once (not per-matmul as PyTorch autograd would)

This is the same technique Unsloth and Axolotl use internally.

In [ ]:
class _FusedLoRAFullFunc(torch.autograd.Function):
    """
    Fully fused: base + LoRA in ONE autograd node so x has a single consumer.
    Produces a single grad_x (no extra buffers from summing multiple grad_x).

    Saves: x, W, A, B (W is param ref — no extra allocation).
    Does NOT save: intermediate A(x) — recomputed in backward.
    """

    @staticmethod
    @torch.amp.custom_fwd(device_type="cuda")
    def forward(ctx, x, W, A, B, scaling):
        base_out = F.linear(x, W)
        inter = F.linear(x, A)
        lora_out = F.linear(inter, B) * scaling
        ctx.save_for_backward(x, W, A, B)
        ctx.scaling = scaling
        return base_out + lora_out

    @staticmethod
    @torch.amp.custom_bwd(device_type="cuda")
    def backward(ctx, grad_output):
        x, W, A, B = ctx.saved_tensors
        scaling = ctx.scaling
        orig_shape = x.shape
        x_2d = x.reshape(-1, x.shape[-1])
        go_2d = grad_output.reshape(-1, grad_output.shape[-1])

        inter_2d = x_2d @ A.t()
        grad_B = scaling * (go_2d.t() @ inter_2d)
        grad_inter = scaling * (go_2d @ B)
        grad_A = grad_inter.t() @ x_2d
        # In-place accumulation to avoid extra temporary (go_2d@W and grad_inter@A)
        grad_x_2d = go_2d @ W
        grad_x_2d.addmm_(grad_inter, A)
        grad_x = grad_x_2d.reshape(orig_shape)
        return grad_x, None, grad_A, grad_B, None


class FusedLoRALinear(nn.Module):
    """
    Drop-in replacement for StandardLoRALinear.
    One autograd node for base+LoRA → one grad_x, lower peak memory.
    """

    def __init__(self, in_features, out_features, rank, alpha, dtype=torch.bfloat16):
        super().__init__()
        self.rank = rank
        self.scaling = alpha / rank

        self.linear = nn.Linear(in_features, out_features, bias=False, dtype=dtype)
        self.linear.weight.requires_grad = False

        self.lora_A = nn.Parameter(torch.empty(rank, in_features, dtype=dtype))
        self.lora_B = nn.Parameter(torch.empty(out_features, rank, dtype=dtype))
        nn.init.kaiming_uniform_(self.lora_A, a=math.sqrt(5))
        nn.init.zeros_(self.lora_B)

    def forward(self, x):
        return _FusedLoRAFullFunc.apply(x, self.linear.weight, self.lora_A, self.lora_B, self.scaling)

## 3. Fused QKV LoRA — Groups Q/K/V into Single Autograd Function

Instead of 3 separate LoRA calls (each saving input `x`), this saves `x` once
and computes all 3 projections in a single forward/backward pass.

In [ ]:
class _FusedLoRAQKVFullFunc(torch.autograd.Function):
    """
    Fully fused Q/K/V: base + LoRA for all 3 in ONE autograd node.
    x has a single consumer → single grad_x (no 4-way sum = less peak memory).
    Saves x once; W's are param refs. Recomputes 3 intermediates in backward.
    """

    @staticmethod
    @torch.amp.custom_fwd(device_type="cuda")
    def forward(ctx, x, W_q, A_q, B_q, s_q, W_k, A_k, B_k, s_k, W_v, A_v, B_v, s_v):
        q = F.linear(x, W_q) + s_q * F.linear(F.linear(x, A_q), B_q)
        k = F.linear(x, W_k) + s_k * F.linear(F.linear(x, A_k), B_k)
        v = F.linear(x, W_v) + s_v * F.linear(F.linear(x, A_v), B_v)
        ctx.save_for_backward(x, W_q, A_q, B_q, W_k, A_k, B_k, W_v, A_v, B_v)
        ctx.scalings = (s_q, s_k, s_v)
        return q, k, v

    @staticmethod
    @torch.amp.custom_bwd(device_type="cuda")
    def backward(ctx, grad_q, grad_k, grad_v):
        x, W_q, A_q, B_q, W_k, A_k, B_k, W_v, A_v, B_v = ctx.saved_tensors
        s_q, s_k, s_v = ctx.scalings
        orig_shape = x.shape
        x_2d = x.reshape(-1, x.shape[-1])

        all_grad_A, all_grad_B = [], []
        grad_x = None  # accumulate in place to reduce peak

        for go, W, A, B, s in [
            (grad_q, W_q, A_q, B_q, s_q),
            (grad_k, W_k, A_k, B_k, s_k),
            (grad_v, W_v, A_v, B_v, s_v),
        ]:
            go_2d = go.reshape(-1, go.shape[-1])
            inter_2d = x_2d @ A.t()
            all_grad_B.append(s * (go_2d.t() @ inter_2d))
            grad_inter = s * (go_2d @ B)
            all_grad_A.append(grad_inter.t() @ x_2d)
            branch = go_2d @ W
            branch.addmm_(grad_inter, A)
            if grad_x is None:
                grad_x = branch
            else:
                grad_x += branch

        grad_x = grad_x.reshape(orig_shape)
        return (
            grad_x,
            None, all_grad_A[0], all_grad_B[0], None,
            None, all_grad_A[1], all_grad_B[1], None,
            None, all_grad_A[2], all_grad_B[2], None,
        )


class FusedLoRAQKV(nn.Module):
    """
    Fully fused Q/K/V: one autograd node for all 3 projections → one grad_x.
    """

    def __init__(self, in_features, out_features_q, out_features_k, out_features_v,
                 rank, alpha, dtype=torch.bfloat16):
        super().__init__()
        self.scaling = alpha / rank

        self.W_q = nn.Linear(in_features, out_features_q, bias=False, dtype=dtype)
        self.W_k = nn.Linear(in_features, out_features_k, bias=False, dtype=dtype)
        self.W_v = nn.Linear(in_features, out_features_v, bias=False, dtype=dtype)
        for m in [self.W_q, self.W_k, self.W_v]:
            m.weight.requires_grad = False

        self.A_q = nn.Parameter(torch.empty(rank, in_features, dtype=dtype))
        self.B_q = nn.Parameter(torch.empty(out_features_q, rank, dtype=dtype))
        self.A_k = nn.Parameter(torch.empty(rank, in_features, dtype=dtype))
        self.B_k = nn.Parameter(torch.empty(out_features_k, rank, dtype=dtype))
        self.A_v = nn.Parameter(torch.empty(rank, in_features, dtype=dtype))
        self.B_v = nn.Parameter(torch.empty(out_features_v, rank, dtype=dtype))

        for A in [self.A_q, self.A_k, self.A_v]:
            nn.init.kaiming_uniform_(A, a=math.sqrt(5))
        for B in [self.B_q, self.B_k, self.B_v]:
            nn.init.zeros_(B)

    def forward(self, x):
        return _FusedLoRAQKVFullFunc.apply(
            x,
            self.W_q.weight, self.A_q, self.B_q, self.scaling,
            self.W_k.weight, self.A_k, self.B_k, self.scaling,
            self.W_v.weight, self.A_v, self.B_v, self.scaling,
        )

## 4. Correctness Tests

Verify that fused and unfused produce identical results (within bf16 tolerance).

In [ ]:
def test_forward_correctness(in_f=4096, out_f=4096, rank=16, alpha=32.0,
                              batch=4, seq_len=1024):
    """Verify FusedLoRALinear produces identical output to StandardLoRALinear."""
    torch.manual_seed(42)

    std = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused = FusedLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    # Copy weights so both modules are identical
    fused.linear.weight.data.copy_(std.linear.weight.data)
    fused.lora_A.data.copy_(std.lora_A.data)
    fused.lora_B.data.copy_(std.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE)

    with torch.no_grad():
        out_std = std(x)
        out_fused = fused(x)

    max_diff = (out_std - out_fused).abs().max().item()
    mean_diff = (out_std - out_fused).abs().mean().item()

    passed = max_diff < 1e-2  # bf16 tolerance
    status = "PASS" if passed else "FAIL"
    print(f"[{status}] Forward correctness: max_diff={max_diff:.6e}, mean_diff={mean_diff:.6e}")
    return passed


def test_backward_correctness(in_f=4096, out_f=4096, rank=16, alpha=32.0,
                               batch=2, seq_len=512):
    """Verify gradients match between fused and unfused."""
    torch.manual_seed(42)

    std = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused = FusedLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    fused.linear.weight.data.copy_(std.linear.weight.data)
    fused.lora_A.data.copy_(std.lora_A.data)
    fused.lora_B.data.copy_(std.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_fused = x.detach().clone().requires_grad_(True)

    # Forward + backward for standard
    out_std = std(x)
    loss_std = out_std.sum()
    loss_std.backward()

    # Forward + backward for fused
    out_fused = fused(x_fused)
    loss_fused = out_fused.sum()
    loss_fused.backward()

    results = []

    # Compare grad_A
    diff_A = (std.lora_A.grad - fused.lora_A.grad).abs().max().item()
    results.append(("grad_A", diff_A))

    # Compare grad_B
    diff_B = (std.lora_B.grad - fused.lora_B.grad).abs().max().item()
    results.append(("grad_B", diff_B))

    # Compare grad_x
    diff_x = (x.grad - x_fused.grad).abs().max().item()
    results.append(("grad_x", diff_x))

    all_passed = True
    for name, diff in results:
        passed = diff < 5e-2  # bf16 gradient tolerance (accumulation error)
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}: max_diff={diff:.6e}")
        if not passed:
            all_passed = False

    print(f"[{'PASS' if all_passed else 'FAIL'}] Backward correctness overall")
    return all_passed


def test_qkv_correctness(in_f=4096, out_f=4096, rank=16, alpha=32.0,
                          batch=2, seq_len=512):
    """Verify FusedLoRAQKV matches 3 separate StandardLoRALinear calls."""
    torch.manual_seed(42)

    # Standard: 3 separate modules
    std_q = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_k = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_v = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    # Fused: single module
    fused = FusedLoRAQKV(in_f, out_f, out_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    # Copy weights
    fused.W_q.weight.data.copy_(std_q.linear.weight.data)
    fused.A_q.data.copy_(std_q.lora_A.data)
    fused.B_q.data.copy_(std_q.lora_B.data)
    fused.W_k.weight.data.copy_(std_k.linear.weight.data)
    fused.A_k.data.copy_(std_k.lora_A.data)
    fused.B_k.data.copy_(std_k.lora_B.data)
    fused.W_v.weight.data.copy_(std_v.linear.weight.data)
    fused.A_v.data.copy_(std_v.lora_A.data)
    fused.B_v.data.copy_(std_v.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE)

    with torch.no_grad():
        q_std = std_q(x)
        k_std = std_k(x)
        v_std = std_v(x)
        q_f, k_f, v_f = fused(x)

    all_passed = True
    for name, a, b in [("Q", q_std, q_f), ("K", k_std, k_f), ("V", v_std, v_f)]:
        diff = (a - b).abs().max().item()
        passed = diff < 1e-2
        status = "PASS" if passed else "FAIL"
        print(f"  [{status}] {name}: max_diff={diff:.6e}")
        if not passed:
            all_passed = False

    print(f"[{'PASS' if all_passed else 'FAIL'}] QKV correctness overall")
    return all_passed


print("=" * 60)
print("CORRECTNESS TESTS")
print("=" * 60)

# Use smaller dims on T4 to avoid OOM
def _gpu_gb():
    if not torch.cuda.is_available():
        return 0
    p = torch.cuda.get_device_properties(0)
    return getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9

gpu_mem = _gpu_gb()
if gpu_mem < 20:  # T4 or similar
    dims = dict(in_f=1024, out_f=1024, batch=2, seq_len=256)
    print(f"Using small dims for {gpu_mem:.0f}GB GPU")
else:
    dims = dict(in_f=4096, out_f=4096, batch=4, seq_len=1024)
    print(f"Using full dims for {gpu_mem:.0f}GB GPU")

print()
test_forward_correctness(**dims)
print()
test_backward_correctness(**dims)
print()
test_qkv_correctness(**dims)

## 5. Memory Benchmark

Compare peak VRAM between fused and unfused for a simulated training step.

In [ ]:
def benchmark_memory(module_fn, x, label, n_warmup=2, n_runs=5):
    """Measure peak memory during forward+backward."""
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()

    # Warmup
    for _ in range(n_warmup):
        out = module_fn(x)
        out.sum().backward()
        torch.cuda.synchronize()

    # Measure
    torch.cuda.empty_cache()
    torch.cuda.reset_peak_memory_stats()
    mem_before = torch.cuda.memory_allocated() / 1e6

    for _ in range(n_runs):
        out = module_fn(x)
        out.sum().backward()
        torch.cuda.synchronize()

    peak = torch.cuda.max_memory_allocated() / 1e6
    current = torch.cuda.memory_allocated() / 1e6

    print(f"  {label}:")
    print(f"    Peak VRAM:    {peak:.1f} MB")
    print(f"    Current VRAM: {current:.1f} MB")
    print(f"    Before VRAM:  {mem_before:.1f} MB")
    return peak


def run_memory_benchmark():
    p = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9
    if gpu_mem < 20:
        in_f, out_f, rank, batch, seq_len = 1024, 1024, 16, 2, 512
    else:
        in_f, out_f, rank, batch, seq_len = 4096, 4096, 16, 4, 1024

    alpha = 32.0
    torch.manual_seed(42)

    print(f"\nConfig: in={in_f}, out={out_f}, rank={rank}, batch={batch}, seq={seq_len}")
    print(f"Input tensor: {batch * seq_len * in_f * 2 / 1e6:.1f} MB (bf16)")
    print()

    # --- Single layer comparison ---
    print("--- Single LoRA Layer ---")
    std = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused = FusedLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused.linear.weight.data.copy_(std.linear.weight.data)
    fused.lora_A.data.copy_(std.lora_A.data)
    fused.lora_B.data.copy_(std.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)

    peak_std = benchmark_memory(std, x, "Standard LoRA")
    x_f = x.detach().clone().requires_grad_(True)
    peak_fused = benchmark_memory(fused, x_f, "Fused LoRA")

    savings = (peak_std - peak_fused) / peak_std * 100
    print(f"\n  Memory savings: {savings:.1f}% ({peak_std - peak_fused:.1f} MB)")

    # --- QKV comparison ---
    print("\n--- QKV LoRA (3 projections) ---")
    del std, fused, x, x_f
    torch.cuda.empty_cache()

    std_q = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_k = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_v = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    fused_qkv = FusedLoRAQKV(in_f, out_f, out_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused_qkv.W_q.weight.data.copy_(std_q.linear.weight.data)
    fused_qkv.A_q.data.copy_(std_q.lora_A.data)
    fused_qkv.B_q.data.copy_(std_q.lora_B.data)
    fused_qkv.W_k.weight.data.copy_(std_k.linear.weight.data)
    fused_qkv.A_k.data.copy_(std_k.lora_A.data)
    fused_qkv.B_k.data.copy_(std_k.lora_B.data)
    fused_qkv.W_v.weight.data.copy_(std_v.linear.weight.data)
    fused_qkv.A_v.data.copy_(std_v.lora_A.data)
    fused_qkv.B_v.data.copy_(std_v.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)

    def std_qkv_forward(x):
        return std_q(x) + std_k(x) + std_v(x)  # sum for scalar loss

    def fused_qkv_forward(x):
        q, k, v = fused_qkv(x)
        return q + k + v

    peak_std = benchmark_memory(std_qkv_forward, x, "Standard 3x LoRA")
    x_f = x.detach().clone().requires_grad_(True)
    peak_fused = benchmark_memory(fused_qkv_forward, x_f, "Fused QKV LoRA")

    savings = (peak_std - peak_fused) / peak_std * 100
    print(f"\n  QKV Memory savings: {savings:.1f}% ({peak_std - peak_fused:.1f} MB)")

    # --- Multi-layer simulation ---
    # This is where savings compound: 70B model has ~80 layers, each with QKV LoRA
    print("\n--- Multi-Layer Simulation (stacked forward+backward) ---")
    del std_q, std_k, std_v, fused_qkv, x, x_f
    torch.cuda.empty_cache()

    n_layers = 8  # simulate 8 layers (limited by T4 VRAM)
    if gpu_mem < 20:
        in_f_ml, out_f_ml, rank_ml, batch_ml, seq_ml = 512, 512, 16, 1, 256
    else:
        in_f_ml, out_f_ml, rank_ml, batch_ml, seq_ml = 2048, 2048, 16, 2, 512

    print(f"  Simulating {n_layers} layers, dims={in_f_ml}, batch={batch_ml}, seq={seq_ml}")

    torch.manual_seed(42)
    std_layers = [StandardLoRALinear(in_f_ml, out_f_ml, rank_ml, alpha, dtype=DTYPE).to(DEVICE) for _ in range(n_layers)]
    fused_layers = []
    for sl in std_layers:
        fl = FusedLoRALinear(in_f_ml, out_f_ml, rank_ml, alpha, dtype=DTYPE).to(DEVICE)
        fl.linear.weight.data.copy_(sl.linear.weight.data)
        fl.lora_A.data.copy_(sl.lora_A.data)
        fl.lora_B.data.copy_(sl.lora_B.data)
        fused_layers.append(fl)

    def multi_std(x):
        h = x
        for layer in std_layers:
            h = layer(h)
        return h

    def multi_fused(x):
        h = x
        for layer in fused_layers:
            h = layer(h)
        return h

    x = torch.randn(batch_ml, seq_ml, in_f_ml, dtype=DTYPE, device=DEVICE, requires_grad=True)
    peak_std_ml = benchmark_memory(multi_std, x, f"Standard x{n_layers} layers")
    x_f = x.detach().clone().requires_grad_(True)
    peak_fused_ml = benchmark_memory(multi_fused, x_f, f"Fused x{n_layers} layers")

    savings_ml = (peak_std_ml - peak_fused_ml) / peak_std_ml * 100
    print(f"\n  Multi-layer savings: {savings_ml:.1f}% ({peak_std_ml - peak_fused_ml:.1f} MB)")
    if savings_ml > 0:
        est_80_layers = (peak_std_ml - peak_fused_ml) / n_layers * 80
        print(f"  Estimated savings at 80 layers (70B scale): ~{est_80_layers:.0f} MB")


print("=" * 60)
print("MEMORY BENCHMARK")
print("=" * 60)
run_memory_benchmark()

## 6. Speed Benchmark

Compare wall-clock time for forward+backward between fused and unfused.

In [ ]:
def benchmark_speed(module_fn, x, label, n_warmup=10, n_runs=50):
    """Measure wall-clock time for forward+backward using CUDA events."""
    torch.cuda.synchronize()

    # Warmup
    for _ in range(n_warmup):
        out = module_fn(x)
        out.sum().backward()
    torch.cuda.synchronize()

    # Timed runs
    start_event = torch.cuda.Event(enable_timing=True)
    end_event = torch.cuda.Event(enable_timing=True)

    start_event.record()
    for _ in range(n_runs):
        out = module_fn(x)
        out.sum().backward()
    end_event.record()
    torch.cuda.synchronize()

    elapsed_ms = start_event.elapsed_time(end_event)
    per_iter_ms = elapsed_ms / n_runs

    print(f"  {label}: {per_iter_ms:.3f} ms/iter ({n_runs} iters, {elapsed_ms:.1f} ms total)")
    return per_iter_ms


def run_speed_benchmark():
    p = torch.cuda.get_device_properties(0)
    gpu_mem = getattr(p, 'total_memory', getattr(p, 'total_mem', 0)) / 1e9
    if gpu_mem < 20:
        in_f, out_f, rank, batch, seq_len = 1024, 1024, 16, 2, 512
    else:
        in_f, out_f, rank, batch, seq_len = 4096, 4096, 16, 4, 1024

    alpha = 32.0
    torch.manual_seed(42)

    print(f"\nConfig: in={in_f}, out={out_f}, rank={rank}, batch={batch}, seq={seq_len}")
    print()

    # --- Single layer ---
    print("--- Single LoRA Layer ---")
    std = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused = FusedLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused.linear.weight.data.copy_(std.linear.weight.data)
    fused.lora_A.data.copy_(std.lora_A.data)
    fused.lora_B.data.copy_(std.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_f = x.detach().clone().requires_grad_(True)

    t_std = benchmark_speed(std, x, "Standard LoRA")
    t_fused = benchmark_speed(fused, x_f, "Fused LoRA")
    speedup = t_std / t_fused
    print(f"  Speedup: {speedup:.2f}x")

    # --- QKV ---
    print("\n--- QKV LoRA (3 projections) ---")
    del std, fused, x, x_f
    torch.cuda.empty_cache()

    std_q = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_k = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    std_v = StandardLoRALinear(in_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)

    fused_qkv = FusedLoRAQKV(in_f, out_f, out_f, out_f, rank, alpha, dtype=DTYPE).to(DEVICE)
    fused_qkv.W_q.weight.data.copy_(std_q.linear.weight.data)
    fused_qkv.A_q.data.copy_(std_q.lora_A.data)
    fused_qkv.B_q.data.copy_(std_q.lora_B.data)
    fused_qkv.W_k.weight.data.copy_(std_k.linear.weight.data)
    fused_qkv.A_k.data.copy_(std_k.lora_A.data)
    fused_qkv.B_k.data.copy_(std_k.lora_B.data)
    fused_qkv.W_v.weight.data.copy_(std_v.linear.weight.data)
    fused_qkv.A_v.data.copy_(std_v.lora_A.data)
    fused_qkv.B_v.data.copy_(std_v.lora_B.data)

    x = torch.randn(batch, seq_len, in_f, dtype=DTYPE, device=DEVICE, requires_grad=True)
    x_f = x.detach().clone().requires_grad_(True)

    def std_qkv_forward(x):
        return std_q(x) + std_k(x) + std_v(x)

    def fused_qkv_forward(x):
        q, k, v = fused_qkv(x)
        return q + k + v

    t_std = benchmark_speed(std_qkv_forward, x, "Standard 3x LoRA")
    t_fused = benchmark_speed(fused_qkv_forward, x_f, "Fused QKV LoRA")
    speedup = t_std / t_fused
    print(f"  QKV Speedup: {speedup:.2f}x")


print("=" * 60)
print("SPEED BENCHMARK")
print("=" * 60)
run_speed_benchmark()

## 7. Summary

Run all tests and print a final summary table.

In [ ]:
print("\n" + "=" * 60)
print("SUMMARY")
print("=" * 60)
print()
print("These fused LoRA kernels use the same techniques as Unsloth:")
print("  1. Custom torch.autograd.Function to control what gets saved")
print("  2. Recompute intermediate (A @ x) in backward instead of storing")
print("  3. Group Q/K/V into single autograd function (save x once, not 3x)")
print()
print("Key differences from Unsloth:")
print("  - Compatible with DeepSpeed ZeRO-3 (Unsloth is NOT)")
print("  - Works with custom model architectures (DeltaNet, GSA, MoE)")
print("  - No dependency on HuggingFace model patterns")
print()
print("Interpreting benchmarks (small dims, e.g. 1024):")
print("  - Correctness: fused matches standard (goal met).")
print("  - Single-layer speed: fused ~1.5x faster (fewer kernels, less Python).")
print("  - QKV speed: fused can be ~same or slightly slower (one big backward).")
print("  - Memory: fused may show negative 'savings' at 1024 dims (2--4 MB).")
print("    The intermediate we avoid is tiny (~32 KB); custom-autograd overhead")
print("    can dominate. At 70B scale (4096+ dims, 80 layers) the design pays off.")
print()
print("Next steps:")
print("  1. Integrate into lora_utils.py as drop-in replacement")
print("  2. Add fused_kernels=true config flag")
print("  3. Test on p4de with 70B model + DeepSpeed ZeRO-3")
print("  4. Optional: add true Triton matmul fusion (further speedup)")